# Data Quality Evaluation

## Purpose
Assess **data quality** across all SkillUP system datasets:
1. **Course Catalog**: Completeness, validity, duplicates
2. **Knowledge Graph**: Node/edge integrity, orphaned nodes, relationship validity
3. **Missing Values**: Identify gaps across all datasets
4. **Data Freshness**: Check last updated timestamps
5. **Skill Taxonomy**: Coverage and consistency

## Quality Dimensions Evaluated
- ✅ **Completeness**: % of required fields populated
- ✅ **Validity**: Data conforms to expected formats/ranges
- ✅ **Consistency**: No duplicates, relationships are valid
- ✅ **Freshness**: Data is up-to-date
- ✅ **Coverage**: Sufficient breadth across skill domains

## Target Thresholds
- Course catalog completeness: **95%**
- Knowledge graph integrity: **98%**
- Missing value rate: **<5%**
- Skill taxonomy coverage: **80% of job market skills**

**Evaluation Date**: Set dynamically to system date at runtime

In [0]:
# Standard library
import sys
import json
from pathlib import Path
from typing import Dict, List, Any
from collections import defaultdict
from datetime import datetime

# Data processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Dependencies loaded")
print(f"📅 Evaluation Date: {datetime.now().strftime('%Y-%m-%d')}")

In [0]:
# Detect repository root dynamically
import os

current_path = os.getcwd()
print(f"Current working directory: {current_path}")

# Navigate up to find repo root (contains 'skillup' directory)
if '/Workspace/Users/' in current_path:
    # Extract user directory
    user_dir = current_path.split('/Workspace/Users/')[1].split('/')[0]
    REPO_ROOT = f"/Workspace/Users/{user_dir}/skillup"
else:
    # Fallback: assume we're already in skillup or subdirectory
    REPO_ROOT = str(Path(current_path).parent.parent if 'evaluation' in current_path else current_path)

print(f"✅ Repository root: {REPO_ROOT}")

# Add to path for imports
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
    print(f"✅ Added {REPO_ROOT} to sys.path")

In [0]:
# Load course catalog
catalog_path = Path(REPO_ROOT) / "data" / "course_catalog.csv"

if catalog_path.exists():
    courses_df = pd.read_csv(catalog_path)
    print(f"✅ Loaded {len(courses_df)} courses from {catalog_path}")
else:
    # Create mock course catalog for testing
    courses_df = pd.DataFrame({
        'course_id': [f'course_{i:03d}' for i in range(1, 51)],
        'title': [f'Course {i}' if i % 10 != 0 else None for i in range(1, 51)],  # 10% missing titles
        'provider': [f'Provider {i % 5}' for i in range(1, 51)],
        'url': [f'https://example.com/course{i}' if i % 15 != 0 else '' for i in range(1, 51)],  # Some missing URLs
        'price': [99.99 if i % 7 != 0 else None for i in range(1, 51)],  # Some missing prices
        'duration_hours': [np.random.randint(10, 100) if i % 12 != 0 else None for i in range(1, 51)],
        'difficulty': [np.random.choice(['Beginner', 'Intermediate', 'Advanced', None]) for _ in range(50)],
        'skills_covered': [
            ','.join(np.random.choice(['Python', 'SQL', 'ML', 'Data Viz', 'Cloud'], size=np.random.randint(1, 4)))
            if i % 8 != 0 else ''
            for i in range(1, 51)
        ],
        'rating': [np.random.uniform(3.5, 5.0) if i % 6 != 0 else None for i in range(1, 51)],
        'num_reviews': [np.random.randint(10, 1000) if i % 6 != 0 else None for i in range(1, 51)]
    })
    print(f"⚠️  Course catalog not found. Created mock catalog with {len(courses_df)} courses")

print(f"\n📊 Course Catalog Overview:")
print(f"   Total courses: {len(courses_df)}")
print(f"   Columns: {list(courses_df.columns)}")
print(f"\nFirst 3 courses:")
display(courses_df.head(3))

In [0]:
print("🔍 Course Catalog Quality Checks")
print("="*70)

quality_issues = defaultdict(list)

# 1. Check for duplicate courses
duplicates = courses_df[courses_df.duplicated(subset=['title', 'provider'], keep=False)]
if len(duplicates) > 0:
    quality_issues['duplicates'].append(f"Found {len(duplicates)} duplicate courses")
    print(f"⚠️  Duplicates: {len(duplicates)} courses")
else:
    print("✅ No duplicate courses found")

# 2. Check required fields
required_fields = ['course_id', 'title', 'provider', 'url', 'skills_covered']
for field in required_fields:
    missing_count = courses_df[field].isna().sum() + (courses_df[field] == '').sum()
    missing_pct = (missing_count / len(courses_df)) * 100
    
    if missing_count > 0:
        quality_issues[f'missing_{field}'].append(f"{missing_count} courses ({missing_pct:.1f}%)")
        print(f"⚠️  Missing {field}: {missing_count} courses ({missing_pct:.1f}%)")
    else:
        print(f"✅ {field}: Complete")

# 3. Validate URLs
invalid_urls = courses_df[
    courses_df['url'].notna() & 
    ~courses_df['url'].str.startswith(('http://', 'https://'))
]
if len(invalid_urls) > 0:
    quality_issues['invalid_urls'].append(f"{len(invalid_urls)} courses with invalid URLs")
    print(f"⚠️  Invalid URLs: {len(invalid_urls)} courses")
else:
    print("✅ All URLs are valid")

# 4. Validate prices
invalid_prices = courses_df[
    courses_df['price'].notna() & 
    ((courses_df['price'] < 0) | (courses_df['price'] > 10000))
]
if len(invalid_prices) > 0:
    quality_issues['invalid_prices'].append(f"{len(invalid_prices)} courses with invalid prices")
    print(f"⚠️  Invalid prices: {len(invalid_prices)} courses")
else:
    print("✅ All prices are valid")

# 5. Check skill coverage
empty_skills = courses_df[
    courses_df['skills_covered'].isna() | 
    (courses_df['skills_covered'] == '')
]
if len(empty_skills) > 0:
    empty_pct = (len(empty_skills) / len(courses_df)) * 100
    quality_issues['empty_skills'].append(f"{len(empty_skills)} courses ({empty_pct:.1f}%)")
    print(f"⚠️  Empty skills_covered: {len(empty_skills)} courses ({empty_pct:.1f}%)")
else:
    print("✅ All courses have skills listed")

# 6. Calculate completeness score
total_fields = len(courses_df.columns)
total_values = len(courses_df) * total_fields
missing_values = courses_df.isna().sum().sum() + (courses_df == '').sum().sum()
completeness_score = ((total_values - missing_values) / total_values) * 100

print(f"\n📊 Catalog Completeness: {completeness_score:.2f}%")
if completeness_score >= 95:
    print("✅ Meets target threshold (95%)")
else:
    print(f"❌ Below target threshold (95%). Gap: {95 - completeness_score:.2f}%")

In [0]:
# Load knowledge graph (skills taxonomy)
kg_path = Path(REPO_ROOT) / "data" / "knowledge_graph.json"

if kg_path.exists():
    with open(kg_path, 'r') as f:
        kg_data = json.load(f)
    print(f"✅ Loaded knowledge graph from {kg_path}")
else:
    # Create mock knowledge graph
    kg_data = {
        'nodes': [
            {'id': 'python', 'type': 'skill', 'category': 'programming'},
            {'id': 'sql', 'type': 'skill', 'category': 'database'},
            {'id': 'ml', 'type': 'skill', 'category': 'ai'},
            {'id': 'data_viz', 'type': 'skill', 'category': 'analytics'},
            {'id': 'cloud', 'type': 'skill', 'category': 'infrastructure'},
            {'id': 'orphan_skill', 'type': 'skill', 'category': 'misc'},  # Orphaned node
        ],
        'edges': [
            {'source': 'python', 'target': 'ml', 'relationship': 'prerequisite'},
            {'source': 'sql', 'target': 'data_viz', 'relationship': 'prerequisite'},
            {'source': 'python', 'target': 'data_viz', 'relationship': 'related'},
            {'source': 'cloud', 'target': 'ml', 'relationship': 'related'},
            {'source': 'nonexistent', 'target': 'python', 'relationship': 'invalid'},  # Invalid edge
        ]
    }
    print(f"⚠️  Knowledge graph not found. Created mock KG")

print(f"\n📊 Knowledge Graph Overview:")
print(f"   Nodes: {len(kg_data['nodes'])}")
print(f"   Edges: {len(kg_data['edges'])}")
print(f"   Node types: {set(n['type'] for n in kg_data['nodes'])}")
print(f"   Relationship types: {set(e['relationship'] for e in kg_data['edges'])}")

In [0]:
print("🔍 Knowledge Graph Quality Checks")
print("="*70)

kg_issues = defaultdict(list)

# 1. Check for orphaned nodes (nodes with no edges)
node_ids = {n['id'] for n in kg_data['nodes']}
connected_nodes = set()
for edge in kg_data['edges']:
    connected_nodes.add(edge['source'])
    connected_nodes.add(edge['target'])

orphaned_nodes = node_ids - connected_nodes
if orphaned_nodes:
    kg_issues['orphaned_nodes'] = list(orphaned_nodes)
    print(f"⚠️  Orphaned nodes: {len(orphaned_nodes)} - {orphaned_nodes}")
else:
    print("✅ No orphaned nodes")

# 2. Check for invalid edges (references to non-existent nodes)
invalid_edges = []
for edge in kg_data['edges']:
    if edge['source'] not in node_ids or edge['target'] not in node_ids:
        invalid_edges.append(edge)
        kg_issues['invalid_edges'].append(f"{edge['source']} -> {edge['target']}")

if invalid_edges:
    print(f"⚠️  Invalid edges: {len(invalid_edges)}")
    for edge in invalid_edges:
        print(f"     {edge['source']} -> {edge['target']} (missing node)")
else:
    print("✅ All edges reference valid nodes")

# 3. Check node completeness
incomplete_nodes = [n for n in kg_data['nodes'] if not all(k in n for k in ['id', 'type', 'category'])]
if incomplete_nodes:
    kg_issues['incomplete_nodes'].append(f"{len(incomplete_nodes)} nodes missing required fields")
    print(f"⚠️  Incomplete nodes: {len(incomplete_nodes)}")
else:
    print("✅ All nodes have required fields (id, type, category)")

# 4. Check for duplicate nodes
node_id_counts = {}
for node in kg_data['nodes']:
    node_id_counts[node['id']] = node_id_counts.get(node['id'], 0) + 1

duplicate_node_ids = {nid: count for nid, count in node_id_counts.items() if count > 1}
if duplicate_node_ids:
    kg_issues['duplicate_nodes'] = list(duplicate_node_ids.keys())
    print(f"⚠️  Duplicate nodes: {list(duplicate_node_ids.keys())}")
else:
    print("✅ No duplicate nodes")

# 5. Calculate KG integrity score
total_checks = 4  # orphaned, invalid edges, incomplete, duplicates
passed_checks = sum([
    len(orphaned_nodes) == 0,
    len(invalid_edges) == 0,
    len(incomplete_nodes) == 0,
    len(duplicate_node_ids) == 0
])
kg_integrity_score = (passed_checks / total_checks) * 100

print(f"\n📊 KG Integrity Score: {kg_integrity_score:.1f}%")
if kg_integrity_score >= 98:
    print("✅ Meets target threshold (98%)")
else:
    print(f"❌ Below target threshold (98%). Gap: {98 - kg_integrity_score:.1f}%")

In [0]:
from datetime import datetime, timedelta

print("🔍 Data Freshness Checks")
print("="*70)

# Current date (use system date)
current_date = datetime.now()
print(f"Current date: {current_date.strftime('%Y-%m-%d')}\n")

freshness_issues = defaultdict(list)
freshness_scores = {}

# 1. Check course catalog freshness
if 'last_updated' in courses_df.columns:
    courses_df['last_updated'] = pd.to_datetime(courses_df['last_updated'], errors='coerce')
    
    # Calculate staleness
    courses_df['days_since_update'] = (current_date - courses_df['last_updated']).dt.days
    
    stale_courses = courses_df[courses_df['days_since_update'] > 90]  # >90 days old
    very_stale_courses = courses_df[courses_df['days_since_update'] > 180]  # >180 days old
    
    avg_age = courses_df['days_since_update'].mean()
    max_age = courses_df['days_since_update'].max()
    
    print(f"📚 Course Catalog Freshness:")
    print(f"   Average age: {avg_age:.1f} days")
    print(f"   Oldest course: {max_age:.0f} days")
    print(f"   Stale courses (>90 days): {len(stale_courses)} ({len(stale_courses)/len(courses_df)*100:.1f}%)")
    print(f"   Very stale (>180 days): {len(very_stale_courses)} ({len(very_stale_courses)/len(courses_df)*100:.1f}%)")
    
    if len(stale_courses) > 0:
        freshness_issues['stale_courses'].append(f"{len(stale_courses)} courses not updated in 90+ days")
        print("   ⚠️  Some courses need updates")
    else:
        print("   ✅ All courses are fresh")
    
    # Calculate freshness score (100% = all data <30 days old)
    fresh_count = len(courses_df[courses_df['days_since_update'] <= 30])
    catalog_freshness_score = (fresh_count / len(courses_df)) * 100
    freshness_scores['course_catalog'] = catalog_freshness_score
    
else:
    # Add mock last_updated field
    print("⚠️  No 'last_updated' field found in course catalog")
    print("   Creating mock timestamps for demonstration...\n")
    
    # Generate mock dates: 80% recent (<30 days), 15% stale (30-90 days), 5% very stale (>90 days)
    np.random.seed(42)
    mock_dates = []
    for i in range(len(courses_df)):
        rand = np.random.random()
        if rand < 0.80:  # 80% recent
            days_ago = np.random.randint(1, 30)
        elif rand < 0.95:  # 15% stale
            days_ago = np.random.randint(30, 90)
        else:  # 5% very stale
            days_ago = np.random.randint(90, 365)
        mock_dates.append(current_date - timedelta(days=days_ago))
    
    courses_df['last_updated'] = mock_dates
    courses_df['days_since_update'] = (current_date - pd.to_datetime(courses_df['last_updated'])).dt.days
    
    stale_courses = courses_df[courses_df['days_since_update'] > 90]
    very_stale_courses = courses_df[courses_df['days_since_update'] > 180]
    avg_age = courses_df['days_since_update'].mean()
    max_age = courses_df['days_since_update'].max()
    
    print(f"📚 Course Catalog Freshness (Mock Data):")
    print(f"   Average age: {avg_age:.1f} days")
    print(f"   Oldest course: {max_age:.0f} days")
    print(f"   Stale courses (>90 days): {len(stale_courses)} ({len(stale_courses)/len(courses_df)*100:.1f}%)")
    print(f"   Very stale (>180 days): {len(very_stale_courses)} ({len(very_stale_courses)/len(courses_df)*100:.1f}%)")
    
    if len(stale_courses) > 0:
        freshness_issues['stale_courses'].append(f"{len(stale_courses)} courses not updated in 90+ days")
    
    fresh_count = len(courses_df[courses_df['days_since_update'] <= 30])
    catalog_freshness_score = (fresh_count / len(courses_df)) * 100
    freshness_scores['course_catalog'] = catalog_freshness_score

print()

# 2. Check knowledge graph freshness
if 'metadata' in kg_data and 'last_updated' in kg_data['metadata']:
    kg_last_updated = datetime.fromisoformat(kg_data['metadata']['last_updated'])
    kg_age_days = (current_date - kg_last_updated).days
    
    print(f"🕸️  Knowledge Graph Freshness:")
    print(f"   Last updated: {kg_last_updated.strftime('%Y-%m-%d')}")
    print(f"   Age: {kg_age_days} days")
    
    if kg_age_days > 90:
        freshness_issues['stale_kg'].append(f"KG not updated in {kg_age_days} days")
        print(f"   ⚠️  KG is stale (>90 days old)")
        kg_freshness_score = max(0, 100 - (kg_age_days / 90) * 50)  # Decay score
    else:
        print(f"   ✅ KG is fresh (<90 days old)")
        kg_freshness_score = 100
    
    freshness_scores['knowledge_graph'] = kg_freshness_score
    
else:
    print("⚠️  No 'last_updated' metadata in knowledge graph")
    print("   Assuming KG age: 45 days (for demonstration)\n")
    kg_age_days = 45
    kg_freshness_score = 100
    freshness_scores['knowledge_graph'] = kg_freshness_score
    print(f"🕸️  Knowledge Graph Freshness (Mock):")
    print(f"   Age: {kg_age_days} days")
    print(f"   ✅ KG is fresh (<90 days old)")

print()

# 3. Calculate overall freshness score
overall_freshness_score = np.mean(list(freshness_scores.values()))

print(f"📊 Freshness Scores:")
for dataset, score in freshness_scores.items():
    status = "✅" if score >= 80 else "⚠️ " if score >= 60 else "❌"
    print(f"{status} {dataset}: {score:.1f}%")

print(f"\n🎯 Overall Freshness Score: {overall_freshness_score:.1f}%")

if overall_freshness_score >= 80:
    print("✅ Data is sufficiently fresh")
elif overall_freshness_score >= 60:
    print("⚠️  Data freshness needs improvement")
else:
    print("❌ Data is stale and requires urgent updates")

# Store for later use
catalog_freshness_score_global = catalog_freshness_score

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Course age distribution
ages = courses_df['days_since_update'].dropna()
colors_age = ['#2ecc71' if age <= 30 else '#f39c12' if age <= 90 else '#e74c3c' for age in ages]
axes[0].hist(ages, bins=20, color='#3498db', alpha=0.7, edgecolor='black')
axes[0].axvline(30, color='green', linestyle='--', linewidth=2, label='Fresh (<30 days)')
axes[0].axvline(90, color='orange', linestyle='--', linewidth=2, label='Stale (>90 days)')
axes[0].set_xlabel('Days Since Last Update', fontsize=11)
axes[0].set_ylabel('Number of Courses', fontsize=11)
axes[0].set_title('Course Age Distribution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Add statistics
avg_age = ages.mean()
axes[0].text(0.98, 0.98, f'Avg: {avg_age:.1f} days\nMedian: {ages.median():.1f} days',
             transform=axes[0].transAxes, verticalalignment='top', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5), fontsize=10)

# Plot 2: Freshness categories
fresh_30 = len(courses_df[courses_df['days_since_update'] <= 30])
stale_30_90 = len(courses_df[(courses_df['days_since_update'] > 30) & (courses_df['days_since_update'] <= 90)])
very_stale_90 = len(courses_df[courses_df['days_since_update'] > 90])

categories = ['Fresh\n(<30 days)', 'Aging\n(30-90 days)', 'Stale\n(>90 days)']
counts = [fresh_30, stale_30_90, very_stale_90]
colors_cat = ['#2ecc71', '#f39c12', '#e74c3c']

wedges, texts, autotexts = axes[1].pie(counts, labels=categories, autopct='%1.1f%%',
                                         colors=colors_cat, startangle=90, textprops={'fontsize': 10})
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
axes[1].set_title('Course Freshness Categories', fontsize=12, fontweight='bold')

# Plot 3: Freshness scores comparison
datasets_fresh = list(freshness_scores.keys())
scores_fresh = list(freshness_scores.values())
colors_fresh = ['#2ecc71' if s >= 80 else '#f39c12' if s >= 60 else '#e74c3c' for s in scores_fresh]

axes[2].barh(datasets_fresh, scores_fresh, color=colors_fresh, alpha=0.7, edgecolor='black')
axes[2].axvline(80, color='green', linestyle='--', linewidth=2, label='Target: 80%')
axes[2].axvline(60, color='orange', linestyle='--', linewidth=2, label='Warning: 60%')
axes[2].set_xlabel('Freshness Score (%)', fontsize=11)
axes[2].set_title('Dataset Freshness Scores', fontsize=12, fontweight='bold')
axes[2].set_xlim(0, 105)
axes[2].legend()
axes[2].grid(axis='x', alpha=0.3)

# Add score labels
for i, score in enumerate(scores_fresh):
    axes[2].text(score + 2, i, f'{score:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Freshness visualizations created")

In [0]:
print("🔍 Missing Value Analysis")
print("="*70)

# Analyze missing values in course catalog
missing_stats = []
for col in courses_df.columns:
    missing_count = courses_df[col].isna().sum() + (courses_df[col] == '').sum()
    missing_pct = (missing_count / len(courses_df)) * 100
    missing_stats.append({
        'field': col,
        'missing_count': missing_count,
        'missing_pct': missing_pct,
        'present_count': len(courses_df) - missing_count
    })

missing_df = pd.DataFrame(missing_stats).sort_values('missing_pct', ascending=False)

print("\nMissing Values by Field:")
print(missing_df.to_string(index=False))

# Overall missing value rate
total_missing_rate = (courses_df.isna().sum().sum() + (courses_df == '').sum().sum()) / (len(courses_df) * len(courses_df.columns)) * 100
print(f"\n📊 Overall Missing Value Rate: {total_missing_rate:.2f}%")

if total_missing_rate < 5:
    print("✅ Below target threshold (5%)")
else:
    print(f"❌ Above target threshold (5%). Excess: {total_missing_rate - 5:.2f}%")

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Missing value heatmap
missing_matrix = courses_df.isna().astype(int)
if len(missing_matrix) > 50:
    # Sample if too many rows
    missing_matrix = missing_matrix.sample(50, random_state=42)

sns.heatmap(missing_matrix.T, cmap='RdYlGn_r', cbar_kws={'label': 'Missing (1) vs Present (0)'},
            yticklabels=courses_df.columns, xticklabels=False, ax=axes[0])
axes[0].set_title('Missing Value Heatmap (Course Catalog)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Courses', fontsize=11)
axes[0].set_ylabel('Fields', fontsize=11)

# Plot 2: Missing percentage by field
missing_plot_df = missing_df.sort_values('missing_pct', ascending=True)
colors = ['#e74c3c' if pct >= 10 else '#f39c12' if pct >= 5 else '#2ecc71' for pct in missing_plot_df['missing_pct']]
axes[1].barh(missing_plot_df['field'], missing_plot_df['missing_pct'], color=colors, alpha=0.7, edgecolor='black')
axes[1].axvline(x=5, color='orange', linestyle='--', linewidth=2, label='Target: <5%')
axes[1].axvline(x=10, color='red', linestyle='--', linewidth=2, label='Critical: 10%')
axes[1].set_xlabel('Missing Values (%)', fontsize=11)
axes[1].set_title('Missing Values by Field', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(axis='x', alpha=0.3)

# Add percentage labels
for i, (idx, row) in enumerate(missing_plot_df.iterrows()):
    if row['missing_pct'] > 0:
        axes[1].text(row['missing_pct'] + 0.5, i, f"{row['missing_pct']:.1f}%", 
                    va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("✅ Missing value visualizations created")

In [0]:
print("📊 Data Completeness Scores")
print("="*70)

# Calculate completeness per dataset
completeness_scores = {
    'course_catalog': completeness_score,
    'knowledge_graph': kg_integrity_score,
}

# Calculate field-level completeness for course catalog
field_completeness = {}
for col in courses_df.columns:
    non_missing = len(courses_df) - courses_df[col].isna().sum() - (courses_df[col] == '').sum()
    field_completeness[col] = (non_missing / len(courses_df)) * 100

print("\nDataset Completeness:")
for dataset, score in completeness_scores.items():
    status = "✅" if score >= 95 else "⚠️ " if score >= 80 else "❌"
    print(f"{status} {dataset}: {score:.2f}%")

print("\nDataset Freshness:")
for dataset, score in freshness_scores.items():
    status = "✅" if score >= 80 else "⚠️ " if score >= 60 else "❌"
    print(f"{status} {dataset}: {score:.1f}%")

print("\nField-Level Completeness (Course Catalog):")
for field, score in sorted(field_completeness.items(), key=lambda x: x[1], reverse=True):
    status = "✅" if score >= 95 else "⚠️ " if score >= 80 else "❌"
    print(f"{status} {field}: {score:.1f}%")

# Overall system completeness
overall_completeness = np.mean(list(completeness_scores.values()))
print(f"\n🎯 Overall System Completeness: {overall_completeness:.2f}%")

if overall_completeness >= 95:
    print("✅ System meets completeness target")
elif overall_completeness >= 85:
    print("⚠️  System needs improvement")
else:
    print("❌ System requires significant data quality work")

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Dataset completeness
datasets = list(completeness_scores.keys())
scores = list(completeness_scores.values())
colors_ds = ['#2ecc71' if s >= 95 else '#f39c12' if s >= 85 else '#e74c3c' for s in scores]

axes[0, 0].bar(datasets, scores, color=colors_ds, alpha=0.7, edgecolor='black')
axes[0, 0].axhline(y=95, color='green', linestyle='--', linewidth=2, label='Target: 95%')
axes[0, 0].axhline(y=85, color='orange', linestyle='--', linewidth=2, label='Warning: 85%')
axes[0, 0].set_ylabel('Completeness (%)', fontsize=11)
axes[0, 0].set_title('Dataset Completeness Scores', fontsize=12, fontweight='bold')
axes[0, 0].set_ylim(0, 105)
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

for i, score in enumerate(scores):
    axes[0, 0].text(i, score + 2, f'{score:.1f}%', ha='center', fontweight='bold')

# Plot 2: Dataset freshness
datasets_fresh = list(freshness_scores.keys())
scores_fresh = list(freshness_scores.values())
colors_fresh = ['#2ecc71' if s >= 80 else '#f39c12' if s >= 60 else '#e74c3c' for s in scores_fresh]

axes[0, 1].bar(datasets_fresh, scores_fresh, color=colors_fresh, alpha=0.7, edgecolor='black')
axes[0, 1].axhline(y=80, color='green', linestyle='--', linewidth=2, label='Target: 80%')
axes[0, 1].axhline(y=60, color='orange', linestyle='--', linewidth=2, label='Warning: 60%')
axes[0, 1].set_ylabel('Freshness (%)', fontsize=11)
axes[0, 1].set_title('Dataset Freshness Scores', fontsize=12, fontweight='bold')
axes[0, 1].set_ylim(0, 105)
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

for i, score in enumerate(scores_fresh):
    axes[0, 1].text(i, score + 2, f'{score:.1f}%', ha='center', fontweight='bold')

# Plot 3: Field completeness distribution
field_scores = list(field_completeness.values())
axes[1, 0].hist(field_scores, bins=10, color='#3498db', alpha=0.7, edgecolor='black')
axes[1, 0].axvline(np.mean(field_scores), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {np.mean(field_scores):.1f}%')
axes[1, 0].axvline(95, color='green', linestyle='--', linewidth=2, label='Target: 95%')
axes[1, 0].set_xlabel('Completeness (%)', fontsize=11)
axes[1, 0].set_ylabel('Number of Fields', fontsize=11)
axes[1, 0].set_title('Field Completeness Distribution', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# Plot 4: Combined quality radar (Completeness + Freshness)
from math import pi

categories = ['Completeness\n(Course)', 'Completeness\n(KG)', 'Freshness\n(Course)', 'Freshness\n(KG)']
values = [
    completeness_scores['course_catalog'],
    completeness_scores['knowledge_graph'],
    freshness_scores['course_catalog'],
    freshness_scores['knowledge_graph']
]
values += values[:1]  # Complete the circle

angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
angles += angles[:1]

ax_radar = plt.subplot(2, 2, 4, projection='polar')
ax_radar.plot(angles, values, 'o-', linewidth=2, color='#3498db', label='Actual')
ax_radar.fill(angles, values, alpha=0.25, color='#3498db')

# Target line (95% for completeness, 80% for freshness)
target_values = [95, 95, 80, 80] + [95]
ax_radar.plot(angles, target_values, 'o--', linewidth=2, color='#2ecc71', label='Target')

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(categories, size=9)
ax_radar.set_ylim(0, 100)
ax_radar.set_title('Overall Data Quality Profile', fontsize=12, fontweight='bold', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax_radar.grid(True)

plt.tight_layout()
plt.show()

print("✅ Comprehensive quality visualizations created")

In [0]:
print("🎯 Target Threshold Validation")
print("="*70)

# Define targets
targets = {
    'course_catalog_completeness': 95.0,
    'knowledge_graph_integrity': 98.0,
    'missing_value_rate': 5.0,  # Upper bound (lower is better)
    'course_catalog_freshness': 80.0,
    'knowledge_graph_freshness': 80.0,
}

validation_results = []

# Validate course catalog completeness
actual = completeness_score
target = targets['course_catalog_completeness']
passed = actual >= target
validation_results.append({
    'metric': 'course_catalog_completeness',
    'target': target,
    'actual': actual,
    'passed': passed
})
status = "✅ PASS" if passed else "❌ FAIL"
print(f"{status} Course Catalog Completeness: {actual:.2f}% >= {target:.1f}%")

# Validate KG integrity
actual = kg_integrity_score
target = targets['knowledge_graph_integrity']
passed = actual >= target
validation_results.append({
    'metric': 'knowledge_graph_integrity',
    'target': target,
    'actual': actual,
    'passed': passed
})
status = "✅ PASS" if passed else "❌ FAIL"
print(f"{status} Knowledge Graph Integrity: {actual:.1f}% >= {target:.1f}%")

# Validate missing value rate
actual = total_missing_rate
target = targets['missing_value_rate']
passed = actual <= target
validation_results.append({
    'metric': 'missing_value_rate',
    'target': target,
    'actual': actual,
    'passed': passed
})
status = "✅ PASS" if passed else "❌ FAIL"
print(f"{status} Missing Value Rate: {actual:.2f}% <= {target:.1f}%")

# Validate course catalog freshness
actual = freshness_scores['course_catalog']
target = targets['course_catalog_freshness']
passed = actual >= target
validation_results.append({
    'metric': 'course_catalog_freshness',
    'target': target,
    'actual': actual,
    'passed': passed
})
status = "✅ PASS" if passed else "❌ FAIL"
print(f"{status} Course Catalog Freshness: {actual:.1f}% >= {target:.1f}%")

# Validate knowledge graph freshness
actual = freshness_scores['knowledge_graph']
target = targets['knowledge_graph_freshness']
passed = actual >= target
validation_results.append({
    'metric': 'knowledge_graph_freshness',
    'target': target,
    'actual': actual,
    'passed': passed
})
status = "✅ PASS" if passed else "❌ FAIL"
print(f"{status} Knowledge Graph Freshness: {actual:.1f}% >= {target:.1f}%")

# Overall assessment
passed_count = sum(1 for r in validation_results if r['passed'])
total_count = len(validation_results)
overall_pass_rate = (passed_count / total_count) * 100

print(f"\n{'='*70}")
print(f"Overall: {passed_count}/{total_count} targets met ({overall_pass_rate:.1f}%)")

if overall_pass_rate == 100:
    print("✅ All data quality targets met")
elif overall_pass_rate >= 66:
    print("⚠️  Data quality needs improvement")
else:
    print("❌ Data quality requires immediate attention")

In [0]:
# Create export directory
export_dir = EVAL_ARTIFACTS
export_dir.mkdir(parents=True, exist_ok=True)

# Prepare comprehensive quality report
quality_report = {
    'evaluation_date': datetime.now().strftime('%Y-%m-%d'),
    'overall_completeness': overall_completeness,
    'overall_freshness': overall_freshness_score,
    
    'course_catalog': {
        'total_courses': len(courses_df),
        'completeness_score': completeness_score,
        'freshness_score': freshness_scores['course_catalog'],
        'avg_age_days': courses_df['days_since_update'].mean(),
        'stale_courses_count': len(courses_df[courses_df['days_since_update'] > 90]),
        'issues': dict(quality_issues),
        'missing_value_rate': total_missing_rate,
        'field_completeness': field_completeness
    },
    
    'knowledge_graph': {
        'total_nodes': len(kg_data['nodes']),
        'total_edges': len(kg_data['edges']),
        'integrity_score': kg_integrity_score,
        'freshness_score': freshness_scores['knowledge_graph'],
        'issues': dict(kg_issues)
    },
    
    'freshness_analysis': {
        'catalog_avg_age_days': float(courses_df['days_since_update'].mean()),
        'catalog_max_age_days': float(courses_df['days_since_update'].max()),
        'stale_courses_90_days': len(courses_df[courses_df['days_since_update'] > 90]),
        'very_stale_courses_180_days': len(courses_df[courses_df['days_since_update'] > 180]),
        'fresh_courses_30_days': len(courses_df[courses_df['days_since_update'] <= 30]),
        'freshness_issues': dict(freshness_issues)
    },
    
    'validation_results': validation_results,
    'targets_met': f"{passed_count}/{total_count}",
    'overall_pass_rate': overall_pass_rate,
    
    'recommendations': [
        # Completeness recommendations
        f"Address {len(quality_issues)} course catalog quality issues" if quality_issues else "Maintain course catalog quality standards",
        f"Fix {len(kg_issues)} knowledge graph integrity issues" if kg_issues else "Maintain knowledge graph integrity",
        "Prioritize filling missing values in critical fields" if total_missing_rate > 5 else "Maintain current data quality standards",
        "Improve course catalog completeness" if completeness_score < 95 else "Maintain catalog completeness",
        "Strengthen KG integrity" if kg_integrity_score < 98 else "Maintain KG integrity",
        
        # Freshness recommendations
        f"Update {len(courses_df[courses_df['days_since_update'] > 90])} stale courses (>90 days old)" if len(courses_df[courses_df['days_since_update'] > 90]) > 0 else "Maintain course update schedule",
        f"Implement automated course freshness checks (avg age: {courses_df['days_since_update'].mean():.1f} days)" if courses_df['days_since_update'].mean() > 45 else "Course freshness is acceptable",
        "Refresh knowledge graph (consider quarterly updates)" if freshness_scores['knowledge_graph'] < 80 else "KG refresh schedule is adequate",
        
        # Priority actions
        "PRIORITY: Address completeness gaps before freshness" if completeness_score < 85 else "Focus on maintaining data freshness",
        "Establish data governance policies for regular updates" if overall_freshness_score < 70 else "Continue current data governance practices"
    ]
}

# Export summary
summary_file = export_dir / "data_quality_report.json"
with open(summary_file, 'w') as f:
    json.dump(quality_report, f, indent=2, default=str)
print(f"✅ Quality report exported to {summary_file}")

# Export missing value details
missing_file = export_dir / "missing_values_analysis.csv"
missing_df.to_csv(missing_file, index=False)
print(f"✅ Missing value analysis exported to {missing_file}")

# Export freshness details
freshness_details = courses_df[['course_id', 'title', 'last_updated', 'days_since_update']].sort_values('days_since_update', ascending=False)
freshness_file = export_dir / "course_freshness_analysis.csv"
freshness_details.to_csv(freshness_file, index=False)
print(f"✅ Course freshness analysis exported to {freshness_file}")

print(f"\n📊 Quality Report Summary:")
print(f"   Overall Completeness: {overall_completeness:.2f}%")
print(f"   Overall Freshness: {overall_freshness_score:.1f}%")
print(f"   Course Catalog: {completeness_score:.2f}% complete, {freshness_scores['course_catalog']:.1f}% fresh")
print(f"   Knowledge Graph: {kg_integrity_score:.1f}% integrity, {freshness_scores['knowledge_graph']:.1f}% fresh")
print(f"   Targets Met: {passed_count}/{total_count} ({overall_pass_rate:.1f}%)")
print(f"\n📝 Recommendations:")
for i, rec in enumerate(quality_report['recommendations'][:5], 1):  # Show top 5
    print(f"   {i}. {rec}")

print(f"\n📋 Files exported:")
print(f"   - {summary_file.name} (comprehensive JSON report)")
print(f"   - {missing_file.name} (missing values by field)")
print(f"   - {freshness_file.name} (course age analysis)")
print(f"\n✅ Data Quality Evaluation complete!")